# NB1 — LangSmith in Production

**Module:** Wayfarer LLMOps Lab &nbsp;•&nbsp; **Type:** Guided

In NB1 we treat LangSmith not as a debugging toy but as a **production control
panel** for the **TripSmith** agent. By the end you will have:

1. **Generated traces** by running TripSmith a few times.
2. **Curated an eval dataset** straight from those production traces.
3. **Built a latency view** — P50 / P95 / P99 — from real run timings.
4. **Created an annotation queue** to route runs to a human reviewer.
5. **Versioned a prompt** and run an **A/B comparison** of two variants.

> Everything here is guarded: if a key is missing the cell prints what to set and
> skips the live call, so you can read and understand without spending tokens.

**Keys used:** `GROQ_API_KEY`, `TAVILY_API_KEY`, `LANGSMITH_API_KEY`,
`LANGSMITH_TRACING=true`.


## 0. Setup

Make the `wayfarer` package importable whether you launch from the repo root or
from `notebooks/`, then load config.


In [6]:
import sys
from pathlib import Path

# Add <repo>/src to the path so `import wayfarer` works from anywhere.
_here = Path.cwd()
for _root in [_here, *_here.parents]:
    if (_root / "src" / "wayfarer").is_dir():
        sys.path.insert(0, str(_root / "src"))
        REPO_ROOT = _root
        break
else:  # running as a script from notebooks/
    REPO_ROOT = Path(__file__).resolve().parents[1]
    sys.path.insert(0, str(REPO_ROOT / "src"))

from wayfarer.config import settings, require_keys, have_key  # noqa: E402
from src.wayfarer.config import AGENT_NAME, COMPANY_NAME, settings  # noqa: E402

print(f"Project        : {settings.langsmith_project}")
print(f"Groq model     : {settings.groq_model}")
print(f"Tracing on?    : {have_key('LANGSMITH_API_KEY') and have_key('LANGSMITH_TRACING')}")


ModuleNotFoundError: No module named 'src.wayfarer'

### How tracing gets turned on

LangSmith tracing for LangChain/LangGraph is **environment-driven** — you do not
wire it into the agent code. Setting these in `.env` is all it takes:

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=wayfarer-tripsmith
```

Every agent invocation then shows up as a **trace** (a tree of runs: the agent, the
LLM calls, the Tavily tool calls) in the LangSmith project. That trace data is the
raw material for everything below.


## 1. Generate some production traces

We run TripSmith on a handful of realistic questions. In a real deployment these
would arrive from users; here we simulate them. Each call emits a trace.


In [ ]:
SAMPLE_QUESTIONS = [
    "Plan a 3-day budget trip to Lisbon in October.",
    "What are the best months to visit Kyoto and why?",
    "I have 5 days in Iceland. Suggest an itinerary with rough costs.",
    "Cheap beach destinations in Southeast Asia for December?",
    "Weekend in Rome: top 3 things to do near the Colosseum.",
]

if require_keys("GROQ_API_KEY", "TAVILY_API_KEY"):
    agent = build_agent(system_prompt=SYSTEM_PROMPT_V1)
    for q in SAMPLE_QUESTIONS:
        answer = ask(agent, q)
        print(f"\nQ: {q}\nA: {answer[:200]}{'…' if len(answer) > 200 else ''}")
    print("\n✅ Traces sent to LangSmith (if tracing is enabled).")
else:
    print("Read on — the curation steps below explain what to do with these traces.")


## 2. Curate an eval dataset from production traces

The single most valuable LangSmith habit: **promote real traces into a dataset** so
you can regression-test against the exact inputs your users send.

The workflow is three calls:

1. `client.list_runs(...)` — pull recent **root** runs (one per user request),
   optionally filtering out errors.
2. `client.create_dataset(...)` — make (or reuse) a dataset.
3. `client.create_examples(...)` — bulk-add `{inputs, outputs}` from those runs.


In [ ]:
from datetime import datetime, timedelta, timezone

DATASET_NAME = "tripsmith-prod-traces"

if require_keys("LANGSMITH_API_KEY"):
    from langsmith import Client

    client = Client()

    # 1. Pull recent successful root runs from our project.
    recent = list(
        client.list_runs(
            project_name=settings.langsmith_project,
            is_root=True,          # one run per user request, not nested LLM/tool steps
            error=False,           # skip failed runs
            start_time=datetime.now(timezone.utc) - timedelta(days=7),
            limit=50,
        )
    )
    print(f"Pulled {len(recent)} recent root runs.")

    if recent:
        # 2. Create the dataset if it doesn't exist yet.
        if client.has_dataset(dataset_name=DATASET_NAME):
            dataset = client.read_dataset(dataset_name=DATASET_NAME)
            print(f"Reusing dataset '{DATASET_NAME}'.")
        else:
            dataset = client.create_dataset(
                dataset_name=DATASET_NAME,
                description="TripSmith inputs/outputs curated from production traces.",
            )
            print(f"Created dataset '{DATASET_NAME}'.")

        # 3. Bulk-add examples. In production you'd hand-pick interesting/failing
        #    cases; here we take the lot.
        examples = [
            {"inputs": r.inputs, "outputs": r.outputs or {}}
            for r in recent
            if r.inputs
        ]
        client.create_examples(dataset_id=dataset.id, examples=examples)
        print(f"Added {len(examples)} examples to '{DATASET_NAME}'.")
    else:
        print("No runs found yet — run Section 1 with valid keys first.")
else:
    print("Showing the pattern only (no LangSmith key set).")


**Teaching note — clean vs raw inputs.** Examples curated this way carry the agent's
*raw* input shape (`{"messages": [...]}`). That's perfect for replaying the exact
scenario, but for clean A/B testing it's often easier to keep a small dataset of
plain questions (`{"question": "..."}`). We build one of those in Section 5.


## 3. Latency dashboard — P50 / P95 / P99

Averages lie. A 200 ms mean can hide a 4-second P99 that's quietly ruining the
experience for 1 in 100 users. The percentiles that matter in production:

| Percentile | Reads as… |
| ---------- | --------- |
| **P50** (median) | the *typical* request |
| **P95** | the *slow* request most users occasionally hit |
| **P99** | the *tail* — your worst-case, SLA-defining latency |

LangSmith shows these in its **Monitor** tab, but computing them ourselves makes the
definition concrete. Each run carries `start_time` and `end_time`; latency is their
difference.


In [ ]:
import numpy as np


def run_latency_seconds(run) -> float | None:
    """Wall-clock latency of a run, in seconds (None if still running)."""
    if run.start_time and run.end_time:
        return (run.end_time - run.start_time).total_seconds()
    return None


latencies: list[float] = []

if require_keys("LANGSMITH_API_KEY"):
    from langsmith import Client

    client = Client()
    runs = list(
        client.list_runs(
            project_name=settings.langsmith_project,
            is_root=True,
            start_time=datetime.now(timezone.utc) - timedelta(days=7),
            limit=200,
        )
    )
    latencies = [s for r in runs if (s := run_latency_seconds(r)) is not None]

if latencies:
    p50, p95, p99 = np.percentile(latencies, [50, 95, 99])
    print(f"n = {len(latencies)} runs")
    print(f"P50: {p50:6.2f}s   P95: {p95:6.2f}s   P99: {p99:6.2f}s")
    print(f"min: {min(latencies):6.2f}s   max: {max(latencies):6.2f}s")
else:
    # Fall back to a synthetic sample so the chart/maths still run offline.
    print("No live latencies — using a synthetic sample to demonstrate the maths.")
    rng = np.random.default_rng(42)
    latencies = np.concatenate(
        [rng.normal(1.2, 0.3, 180), rng.normal(4.5, 1.0, 20)]  # 10% slow tail
    ).clip(0.1).tolist()
    p50, p95, p99 = np.percentile(latencies, [50, 95, 99])
    print(f"P50: {p50:.2f}s   P95: {p95:.2f}s   P99: {p99:.2f}s")


In [ ]:
# Visualise the distribution with the percentile lines marked.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies, bins=30, alpha=0.7, edgecolor="white")
for label, value, color in [("P50", p50, "green"), ("P95", p95, "orange"), ("P99", p99, "red")]:
    ax.axvline(value, color=color, linestyle="--", linewidth=2, label=f"{label} = {value:.2f}s")
ax.set_xlabel("Latency (seconds)")
ax.set_ylabel("Number of requests")
ax.set_title("TripSmith request latency distribution")
ax.legend()
fig.tight_layout()

_out = REPO_ROOT / "notebooks" / "nb1_latency_percentiles.png"
fig.savefig(_out, dpi=110)
print(f"Saved chart → {_out}")
plt.show()


**What to do with this:** wire an alert when P95 crosses your budget (say 3 s). In
LangSmith you'd set this on the Monitor dashboard; the principle is the same — watch
the tail, not the mean.


## 4. Annotation queues — humans in the loop

Automated evals catch a lot, but some judgments need a person ("was this itinerary
actually sensible?"). An **annotation queue** is a review inbox: you route selected
runs into it, and reviewers score them against a rubric in the LangSmith UI.

A common pattern: **queue the slow or low-confidence runs** for human eyes.


In [ ]:
QUEUE_NAME = "tripsmith-human-review"

if require_keys("LANGSMITH_API_KEY"):
    from langsmith import Client

    client = Client()

    # Create the queue (idempotent-ish: reuse by name if it exists).
    existing = [q for q in client.list_annotation_queues() if q.name == QUEUE_NAME]
    if existing:
        queue = existing[0]
        print(f"Reusing annotation queue '{QUEUE_NAME}'.")
    else:
        queue = client.create_annotation_queue(
            name=QUEUE_NAME,
            description="Human review of TripSmith itineraries.",
        )
        print(f"Created annotation queue '{QUEUE_NAME}'.")

    # Pick which runs to review — here, the slowest few from the last 7 days.
    runs = list(
        client.list_runs(
            project_name=settings.langsmith_project,
            is_root=True,
            start_time=datetime.now(timezone.utc) - timedelta(days=7),
            limit=200,
        )
    )
    slowest = sorted(
        (r for r in runs if run_latency_seconds(r) is not None),
        key=run_latency_seconds,
        reverse=True,
    )[:5]

    if slowest:
        client.add_runs_to_annotation_queue(
            queue.id, run_ids=[r.id for r in slowest]
        )
        print(f"Queued {len(slowest)} slow runs for human review.")
    else:
        print("No runs to queue yet — generate some in Section 1 first.")
else:
    print("Annotation-queue pattern shown above (no LangSmith key set).")


Reviewers now open the queue in LangSmith, score each run, and their feedback can
feed back into your dataset (good cases become regression tests; bad cases become
the failures you fix next).


## 5. Prompt versioning + A/B comparison

Prompts are code. Treat them like it: **version them**, and **measure** before you
promote a change. LangSmith's Prompt Hub gives every push a commit hash, so prod can
pin an exact version instead of a moving "latest".

We'll (a) push two prompt variants, then (b) run both over a small clean dataset and
compare a simple score.


In [ ]:
# (a) Push both prompt variants to the Prompt Hub.
from langchain_core.prompts import ChatPromptTemplate

prompt_v1 = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_PROMPT_V1), ("placeholder", "{messages}")]
)
prompt_v2 = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_PROMPT_V2), ("placeholder", "{messages}")]
)

if require_keys("LANGSMITH_API_KEY"):
    from langsmith import Client

    client = Client()
    url_v1 = client.push_prompt("tripsmith-system", object=prompt_v1, description="v1 baseline")
    url_v2 = client.push_prompt("tripsmith-system", object=prompt_v2, description="v2 structured + grounded")
    print("Pushed two commits of 'tripsmith-system'. The latest URL:")
    print(url_v2)
    print("\nPin a specific version in prod with:  client.pull_prompt('tripsmith-system:<commit_hash>')")
else:
    print("Prompt-push pattern shown (no LangSmith key set).")


In [ ]:
# (b) Build a small CLEAN eval dataset of plain questions (easy to replay + score).
AB_DATASET = "tripsmith-ab-questions"

AB_QUESTIONS = [
    {"question": "Plan a 2-day trip to Porto on a tight budget."},
    {"question": "Best season to see cherry blossoms in Tokyo?"},
    {"question": "3 family-friendly activities in Barcelona."},
    {"question": "A relaxed 4-day itinerary for the Amalfi Coast."},
]

if require_keys("LANGSMITH_API_KEY"):
    from langsmith import Client

    client = Client()
    if not client.has_dataset(dataset_name=AB_DATASET):
        ds = client.create_dataset(dataset_name=AB_DATASET, description="Clean A/B questions.")
        client.create_examples(dataset_id=ds.id, examples=AB_QUESTIONS)
        print(f"Created '{AB_DATASET}' with {len(AB_QUESTIONS)} examples.")
    else:
        print(f"Reusing '{AB_DATASET}'.")


Now define a **target** per variant (run the agent with that prompt) and a simple
**evaluator**. We use a cheap heuristic so the comparison runs offline-friendly; in
practice you'd add an LLM-as-judge for answer quality.


In [ ]:
def make_target(system_prompt: str):
    """Return a function evaluate() can call: inputs -> outputs."""
    agent = build_agent(system_prompt=system_prompt)

    def target(inputs: dict) -> dict:
        return {"answer": ask(agent, inputs["question"])}

    return target


def answers_with_budget(outputs: dict, **kwargs) -> dict:
    """Heuristic: did the answer mention a budget/cost? (v2 should do this more.)"""
    text = (outputs or {}).get("answer", "").lower()
    hit = any(w in text for w in ["budget", "cost", "$", "€", "£", "per day", "approx"])
    return {"key": "mentions_budget", "score": int(hit)}


def nonempty(outputs: dict, **kwargs) -> dict:
    text = (outputs or {}).get("answer", "").strip()
    return {"key": "nonempty", "score": int(bool(text))}


In [ ]:
# Run the A/B. Two experiments over the same dataset, then compare mean scores.
if require_keys("GROQ_API_KEY", "TAVILY_API_KEY", "LANGSMITH_API_KEY"):
    from langsmith import evaluate

    res_v1 = evaluate(
        make_target(SYSTEM_PROMPT_V1),
        data=AB_DATASET,
        evaluators=[nonempty, answers_with_budget],
        experiment_prefix="tripsmith-v1",
    )
    res_v2 = evaluate(
        make_target(SYSTEM_PROMPT_V2),
        data=AB_DATASET,
        evaluators=[nonempty, answers_with_budget],
        experiment_prefix="tripsmith-v2",
    )

    def mean_score(results, key: str) -> float:
        df = results.to_pandas()
        col = f"feedback.{key}"
        return float(df[col].mean()) if col in df else float("nan")

    print("            nonempty   mentions_budget")
    print(f"v1 (base)     {mean_score(res_v1, 'nonempty'):.2f}        {mean_score(res_v1, 'mentions_budget'):.2f}")
    print(f"v2 (struct.)  {mean_score(res_v2, 'nonempty'):.2f}        {mean_score(res_v2, 'mentions_budget'):.2f}")
    print("\nOpen LangSmith → Experiments to see the side-by-side comparison view.")
else:
    print("A/B run needs Groq + Tavily + LangSmith keys. Pattern shown above.")


**Reading the result:** if v2 reliably scores higher on `mentions_budget` without
hurting anything else, that's your evidence to promote it — and because it's
versioned, promotion is just pinning the v2 commit hash in production config.


## Recap

You now have the four production LangSmith muscles:

* **Datasets from traces** — regression-test on real user inputs.
* **Latency percentiles** — watch the tail (P95/P99), not the mean.
* **Annotation queues** — route hard cases to humans.
* **Prompt versioning + A/B** — measure changes before promoting them.

**Next — NB2:** we wrap this exact TripSmith agent in a FastAPI service
(`POST /invocations`, `GET /ping`) with async SQLite connection pooling and
automatic tracing, built to the Amazon Bedrock AgentCore contract so the same
container runs locally, in Docker, and on AWS.
